# QEA-ANFIS End-to-End Pipeline

Wires together the modules in `src/` (clone this repo onto Kaggle, or copy `src/` into
`/kaggle/working/src` and add it to `sys.path`). See `README.md` for the summary of what's
validated and `docs/findings_log.md` for the full numeric trail behind every claim below.


In [ ]:
!pip install -q scipy scikit-learn statsmodels tqdm shap xgboost mne autoreject


In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath('src'))  # adjust if src/ lives elsewhere on Kaggle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from preprocessing import (load_raw, preprocess_all, method_A_none, method_B_p2p, method_C_mad,
                            old_global_z_reject, qc_summary, CHANNEL_NAMES, FS)
from features import extract_all_features, run_fuzzy_entropy_unit_tests
from anfis import ANFIS, run_anfis_sanity_check
from ga_optimizer import ga_optimize, select_features
from cv import nested_subject_cv_v2, run_baselines, compute_metrics
from stats import paired_stat_tests, quick_paired
from explainability import extract_top_rules, shap_agreement_analysis, feature_selection_stability

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N_OUTER = 10
OUT_DIR = "/kaggle/working/qea_anfis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Setup complete.")


## 1. Load + verify raw data (Rule 3)

In [ ]:
DATA_DIR = "/kaggle/input/datasets/abinayajone/stew-mental-workload"  # EDIT if needed
X_raw, metadata_raw = load_raw(DATA_DIR)
print("X_raw:", X_raw.shape, "| subjects:", metadata_raw['subject'].nunique())


## 2. ANFIS sanity check -- run this BEFORE trusting any model result

In [ ]:
anfis_check = run_anfis_sanity_check(random_state=RANDOM_SEED)
print(anfis_check)
assert abs(anfis_check['n_rules=1'] - anfis_check['plain_ols_reference']) < 1e-6
print("ANFIS sanity check PASSED.")


## 3. FuzzyEn unit tests -- run before trusting feature extraction

In [ ]:
fe_results = run_fuzzy_entropy_unit_tests()
print(fe_results)
print("FuzzyEn unit tests PASSED.")


## 4. Filtering + artifact QC (Rules 2, 4, 5)

Method C (per-channel robust MAD) is the validated final method -- 16.24% rejection on the full
dataset, all 48 subjects retained. See `docs/findings_log.md` §2-3 for why the original global
z-score criterion rejected 92.4% and why Method C fixes it.

In [ ]:
X_filt = preprocess_all(X_raw, apply_notch=False)  # notch confirmed redundant, see findings_log.md

keep_A = method_A_none(X_filt)
keep_B = method_B_p2p(X_filt)
keep_C = method_C_mad(X_filt)
keep_OLD = ~old_global_z_reject(X_filt)  # for ablation A2 only -- NOT the method used downstream

qc_table = pd.DataFrame([
    qc_summary('A: none', keep_A, metadata_raw),
    qc_summary('B: p2p', keep_B, metadata_raw),
    qc_summary('C: MAD (FINAL)', keep_C, metadata_raw),
])
print(qc_table)
print(f"\nOld criterion (for A2 ablation only): {(~keep_OLD).sum()}/{len(keep_OLD)} rejected "
      f"({100*(~keep_OLD).mean():.1f}%)")

keep_final = keep_C
qc_table.to_csv(f"{OUT_DIR}/table2_artifact_processing_statistics.csv", index=False)


## 5. Feature extraction (Rule 6) -- the slow step, ~12-14 min for ~12k epochs

In [ ]:
X_final, y_final, groups_final, FEATURE_NAMES = extract_all_features(X_filt, keep_final, metadata_raw)
print("X_final:", X_final.shape, "| class balance:", dict(zip(*np.unique(y_final, return_counts=True))))

np.save(f"{OUT_DIR}/X_final.npy", X_final)
np.save(f"{OUT_DIR}/y_final.npy", y_final)
np.save(f"{OUT_DIR}/groups_final.npy", groups_final)
pd.Series(FEATURE_NAMES).to_csv(f"{OUT_DIR}/feature_names.csv", index=False, header=['feature_name'])
print("Saved to", OUT_DIR)


## 6. Main result: nested subject-wise CV + baselines (Rules 11-13)

`nested_subject_cv_v2` includes the SelectKBest(k=60) pre-filter (fit on inner-train only) that
fixed a GA-overfitting problem found during development -- see `docs/findings_log.md` §6 for why
this is necessary, not optional.

In [ ]:
t0 = time.time()
baseline_df = run_baselines(X_final, y_final, groups_final, n_outer=N_OUTER, random_state=RANDOM_SEED)
print(f"Baselines: {(time.time()-t0)/60:.1f} min")
baseline_df.to_csv(f"{OUT_DIR}/baseline_df.csv", index=False)

t0 = time.time()
nested_df, fold_details = nested_subject_cv_v2(
    X_final, y_final, groups_final, FEATURE_NAMES, n_outer=N_OUTER, prefilter_k=60,
    ga_kwargs={'pop_size': 30, 'n_generations': 20, 'feature_penalty_w': 0.05, 'instability_w': 0.1},
    random_state=RANDOM_SEED)
print(f"QEA-ANFIS nested CV: {(time.time()-t0)/60:.1f} min")
nested_df.to_csv(f"{OUT_DIR}/nested_df.csv", index=False)

print("\nQEA-ANFIS:", nested_df[['f1','balanced_accuracy','roc_auc']].mean().to_dict())
print("Baselines:\n", baseline_df.groupby('model')[['f1','balanced_accuracy','roc_auc']].mean())


## 7. Statistical testing (Rule 16)

In [ ]:
stat_results = paired_stat_tests(nested_df, baseline_df, metric='f1')
print(stat_results)
stat_results.to_csv(f"{OUT_DIR}/table7_statistical_tests.csv", index=False)


## 8. Ablation matrix (Rule 14)

A1 (no QC) and A2 (old criterion) reuse features extracted on the FULL 14,208-epoch set -- extract
once here, then mask, rather than re-running feature extraction per ablation.

In [ ]:
kept_idx_all = np.arange(X_filt.shape[0])
X_all_feat, y_all, groups_all, _ = extract_all_features(X_filt, np.ones(X_filt.shape[0], dtype=bool), metadata_raw)
np.save(f"{OUT_DIR}/X_all_feat.npy", X_all_feat)
print("X_all_feat:", X_all_feat.shape)


In [ ]:
feature_names_arr = np.array(FEATURE_NAMES)
spectral_mask = np.array(['abspow_' in n or 'relpow_' in n or 'logpow_' in n or n.startswith('ratio_')
                           or n.startswith('asym_') for n in feature_names_arr])
spectral_nonlinear_mask = spectral_mask | np.array(['fuzzyen_' in n for n in feature_names_arr])

def run_ablation(name, X, y, groups, prefilter_k=60, ga_gens=20, ga_extra=None, n_outer=N_OUTER):
    ga_kwargs = {'pop_size': 30, 'n_generations': ga_gens, 'feature_penalty_w': 0.05, 'instability_w': 0.1}
    if ga_extra:
        ga_kwargs.update(ga_extra)
    print(f"\n=== {name} (n={X.shape[0]}, {X.shape[1]} feats, n_outer={n_outer}) ===")
    t0 = time.time()
    try:
        df, fd = nested_subject_cv_v2(X, y, groups, list(range(X.shape[1])), n_outer=n_outer,
                                       prefilter_k=min(prefilter_k, X.shape[1]), ga_kwargs=ga_kwargs,
                                       random_state=RANDOM_SEED)
    except ValueError as e:
        print(f"  FAILED at n_outer={n_outer}: {e}. Retrying at n_outer=5.")
        df, fd = nested_subject_cv_v2(X, y, groups, list(range(X.shape[1])), n_outer=5,
                                       prefilter_k=min(prefilter_k, X.shape[1]), ga_kwargs=ga_kwargs,
                                       random_state=RANDOM_SEED)
    print(f"{name}: {(time.time()-t0)/60:.1f} min | F1={df['f1'].mean():.3f}+-{df['f1'].std():.3f} | "
          f"n_feat={df['n_selected_features'].mean():.1f}")
    return df, fd

ablations = {}
ablations['A1_no_qc'], _ = run_ablation('A1_no_artifact_handling', X_all_feat, y_all, groups_all)
ablations['A2_old_zscore'], _ = run_ablation('A2_old_global_zscore', X_all_feat[keep_OLD], y_all[keep_OLD], groups_all[keep_OLD])
ablations['A5_feat_select_only'], _ = run_ablation('A5_feature_select_only', X_final, y_final, groups_final,
                                                     ga_extra={'premise_fixed': True})
ablations['A6_premise_only'], _ = run_ablation('A6_premise_only', X_final, y_final, groups_final,
                                                 ga_extra={'mask_fixed': True})
ablations['A8_spectral_only'], _ = run_ablation('A8_spectral_only', X_final[:, spectral_mask], y_final, groups_final)
ablations['A9_spectral_nonlinear'], _ = run_ablation('A9_spectral_plus_nonlinear',
                                                        X_final[:, spectral_nonlinear_mask], y_final, groups_final)

for name, df in ablations.items():
    df.to_csv(f"{OUT_DIR}/ablation_{name}.csv", index=False)

print("\n--- Key ablation comparisons ---")
quick_paired(nested_df, ablations['A1_no_qc'], 'Joint(QC)', 'A1(no QC)')
quick_paired(nested_df, ablations['A5_feat_select_only'], 'Joint', 'A5(feat-select-only)')
quick_paired(nested_df, ablations['A6_premise_only'], 'Joint', 'A6(premise-only)')
quick_paired(ablations['A6_premise_only'], ablations['A5_feat_select_only'], 'A6', 'A5')


## 9. Explainability (Rule 17)

In [ ]:
stability_df = feature_selection_stability(fold_details, FEATURE_NAMES, top_n=20)
print(stability_df)
stability_df.to_csv(f"{OUT_DIR}/table8_feature_stability.csv", index=False)

best_fold_num = nested_df.loc[nested_df['f1'].idxmax(), 'fold']
best_fd = [fd for fd in fold_details if fd['fold'] == best_fold_num][0]
best_model = best_fd['model']
best_feature_names = [FEATURE_NAMES[i] for i in best_fd['selected_original_idx']]

from sklearn.model_selection import StratifiedGroupKFold
outer_cv_check = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_SEED)
splits = list(outer_cv_check.split(X_final, y_final, groups_final))
train_idx_best, test_idx_best = splits[best_fold_num]
scaler_best = best_fd['scaler']
sel_local_idx = np.where(best_fd['ga_result']['feature_mask'] == 1)[0]
X_te_best_s = scaler_best.transform(X_final[test_idx_best])[:, best_fd['prefilter_idx']][:, sel_local_idx]

rules = extract_top_rules(best_model, best_feature_names, X_te_best_s)
for r in rules:
    print(f"Rule {r['rule_idx']} (avg firing={r['avg_firing']:.3f}): {r['top_terms']}")
pd.DataFrame(rules).to_csv(f"{OUT_DIR}/table8b_top_rules.csv", index=False)

X_tr_best_s = scaler_best.transform(X_final[train_idx_best])
shap_values, rf_ref, agreement, top_shap_names = shap_agreement_analysis(
    X_tr_best_s, y_final[train_idx_best], scaler_best.transform(X_final[test_idx_best]),
    FEATURE_NAMES, best_feature_names,
    [t.split('(')[0] for r in rules for t in r['top_terms'].split(', ')],
    random_state=RANDOM_SEED)
print(agreement)


## 10. Reproducibility export (Rule 18)

In [ ]:
import pickle
with open(f"{OUT_DIR}/fold_details.pkl", 'wb') as f:
    pickle.dump(fold_details, f)

bundle = [{'fold': fd['fold'], 'n_selected_features': fd['ga_result']['n_selected_features'],
           'selected_feature_names': [FEATURE_NAMES[i] for i in fd['selected_original_idx']],
           'best_fitness': fd['ga_result']['best_fitness'],
           'fitness_history': fd['ga_result']['fitness_history']} for fd in fold_details]
with open(f"{OUT_DIR}/reproducibility_bundle.json", 'w') as f:
    json.dump(bundle, f, indent=2, default=str)

print("All files in OUT_DIR:")
for fn in sorted(os.listdir(OUT_DIR)):
    print(" ", fn)
